<a href="https://colab.research.google.com/github/HopeSilkina/deposits_forecast_project/blob/main/notebooks/05_SARIMA_Modeling_Deposits_Forecast_Ru.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# БЛОК 5: МОДЕЛИ ВРЕМЕННЫХ РЯДОВ (SARIMA)
# Проект: Прогнозирование объема вкладов населения РФ
# Автор: Надежда Силкина
# Дата: 2026
# ============================================================

# ============================================================
# 1. ПОДКЛЮЧЕНИЕ БИБЛИОТЕК
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from itertools import product
import warnings
warnings.filterwarnings('ignore')

# Настройка графиков
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Библиотеки загружены")

# ============================================================
# 2. ЗАГРУЗКА ДАННЫХ
# ============================================================

url = 'https://raw.githubusercontent.com/HopeSilkina/deposits_forecast_project/main/data/processed_deposits_data.xlsx'
df = pd.read_excel(url, sheet_name='data')
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

print(f"✅ Данные загружены. Записей: {len(df)}")
print(f"Период: с {df.index.min()} по {df.index.max()}")

# ============================================================
# 3. МЕТОДОЛОГИЧЕСКОЕ ОБОСНОВАНИЕ ВЫБОРА SARIMA
# ============================================================

print("\n" + "="*60)
print("3. ПОЧЕМУ SARIMA, А НЕ PROPHET")
print("="*60)

print("""
📌 КЛЮЧЕВАЯ ПРОБЛЕМА: АВТОКОРРЕЛЯЦИЯ ОСТАТКОВ 4-ГО ПОРЯДКА

Во всех предыдущих моделях (базовая Ridge, полная модель с 38 признаками)
диагностика остатков на обучающей выборке выявила значимую автокорреляцию
4-го порядка (тест Бройша-Годфри, p < 0.05).

ПОЧЕМУ НЕ PROPHET:
- Prophet не учитывает автокорреляцию в остатках напрямую
- При автокоррелированных ошибках дает смещенные прогнозы

ПОЧЕМУ SARIMA:
- Явно моделирует автокорреляцию через AR/MA компоненты
- Сезонные компоненты могут учесть квартальную/годовую структуру
- Ljung-Box тест позволяет проверить, устранена ли автокорреляция

ВЫВОД: Выбираем SARIMA для решения проблемы автокорреляции.
""")

# ============================================================
# 4. АНАЛИЗ СТАЦИОНАРНОСТИ РЯДА DEPOS
# ============================================================

print("\n" + "="*60)
print("4. АНАЛИЗ СТАЦИОНАРНОСТИ РЯДА DEPOS")
print("="*60)

print("""
📌 ПРОВЕРЯЕМЫЙ РЯД: DEPOS (объем вкладов населения РФ, млрд руб.)

Два теста с разными гипотезами:
- ADF: H₀ = ряд НЕстационарный (есть единичный корень)
- KPSS: H₀ = ряд стационарный (нет единичного корня)
""")

# 4.1. Тест ADF (константа, без тренда)
print("\n🔍 Тест Дики-Фуллера (ADF):")
print("   H₀: ряд нестационарный | H₁: ряд стационарный")
adf_result = adfuller(df['DEPOS'], autolag='AIC', regression='c')
print(f"   ADF-статистика: {adf_result[0]:.4f}")
print(f"   p-значение: {adf_result[1]:.4f}")
print(f"   Критические значения: 1%: {adf_result[4]['1%']:.4f}, 5%: {adf_result[4]['5%']:.4f}")
print(f"   Вывод: {'Стационарный ✅' if adf_result[1] < 0.05 else 'Нестационарный ❌ (p > 0.05, не отвергаем H₀)'}")

# 4.2. Тест KPSS (константа + тренд)
print("\n🔍 Тест KPSS:")
print("   H₀: ряд стационарный | H₁: ряд нестационарный")
kpss_result = kpss(df['DEPOS'], regression='ct')
print(f"   KPSS-статистика: {kpss_result[0]:.4f}")
print(f"   p-значение: {kpss_result[1]:.4f}")
print(f"   Критические значения: 1%: {kpss_result[3]['1%']:.4f}, 5%: {kpss_result[3]['5%']:.4f}")
print(f"   Вывод: {'Стационарный ✅' if kpss_result[1] > 0.05 else 'Нестационарный ❌ (p < 0.05, отвергаем H₀)'}")

print("\n📌 ОБЩИЙ ВЫВОД: Оба теста подтверждают, что ряд DEPOS НЕСТАЦИОНАРНЫЙ")
print("   → Требуется дифференцирование")

# 4.3. График ряда
plt.figure(figsize=(14, 6))
plt.plot(df.index, df['DEPOS'], linewidth=2, color='steelblue')
plt.title('Ряд DEPOS: Объем вкладов населения РФ', fontsize=14)
plt.xlabel('Дата')
plt.ylabel('млрд руб.')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('05_depos_series.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================================
# 5. ОПРЕДЕЛЕНИЕ ПОРЯДКА ДИФФЕРЕНЦИРОВАНИЯ (d)
# ============================================================

print("\n" + "="*60)
print("5. ОПРЕДЕЛЕНИЕ ПОРЯДКА ДИФФЕРЕНЦИРОВАНИЯ (d)")
print("="*60)

print("""
📌 МЕТОДИКА: Последовательно проверяем разности ряда на стационарность.
d=0 → если нестационарен, d=1 → если нестационарен, d=2.
""")

# --- Проверка первой разности (d=1) ---
print("\n🔍 Проверка ПЕРВОЙ разности (d=1):")
df['DEPOS_diff1'] = df['DEPOS'].diff()

adf_diff1 = adfuller(df['DEPOS_diff1'].dropna(), autolag='AIC', regression='c')
kpss_diff1 = kpss(df['DEPOS_diff1'].dropna(), regression='c')

print(f"   ADF: stat = {adf_diff1[0]:.4f}, p = {adf_diff1[1]:.4f}")
print(f"   KPSS: stat = {kpss_diff1[0]:.4f}, p = {kpss_diff1[1]:.4f}")

d1_stationary = (adf_diff1[1] < 0.05) and (kpss_diff1[1] > 0.05)

if d1_stationary:
    print(f"   ✅ Первая разность СТАЦИОНАРНА → d = 1")
    d_order = 1
else:
    print(f"   ❌ Первая разность НЕСТАЦИОНАРНА (ADF p={adf_diff1[1]:.4f} > 0.05)")
    print(f"   → Проверяем вторую разность")

    # --- Проверка второй разности (d=2) ---
    print("\n🔍 Проверка ВТОРОЙ разности (d=2):")
    df['DEPOS_diff2'] = df['DEPOS_diff1'].diff()

    adf_diff2 = adfuller(df['DEPOS_diff2'].dropna(), autolag='AIC', regression='c')
    kpss_diff2 = kpss(df['DEPOS_diff2'].dropna(), regression='c')

    print(f"   ADF: stat = {adf_diff2[0]:.4f}, p = {adf_diff2[1]:.4f}")
    print(f"   KPSS: stat = {kpss_diff2[0]:.4f}, p = {kpss_diff2[1]:.4f}")

    d2_stationary = (adf_diff2[1] < 0.05) and (kpss_diff2[1] > 0.05)

    if d2_stationary:
        print(f"   ✅ Вторая разность СТАЦИОНАРНА → d = 2")
        d_order = 2
    else:
        print(f"   ⚠️ Даже вторая разность нестационарна (необычно для экономических рядов)")
        d_order = 2  # обычно достаточно

# Графики разностей
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

axes[0].plot(df.index, df['DEPOS'], linewidth=1.5, color='steelblue')
axes[0].set_title('Исходный ряд (d=0)')
axes[0].set_ylabel('DEPOS')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df.index, df['DEPOS_diff1'], linewidth=1.5, color='darkorange')
axes[1].set_title(f'Первая разность (d=1): ADF p={adf_diff1[1]:.4f}, KPSS p={kpss_diff1[1]:.4f}')
axes[1].set_ylabel('Δ DEPOS')
axes[1].grid(True, alpha=0.3)

if d_order == 2:
    axes[2].plot(df.index, df['DEPOS_diff2'], linewidth=1.5, color='green')
    axes[2].set_title(f'Вторая разность (d=2): ADF p={adf_diff2[1]:.4f}, KPSS p={kpss_diff2[1]:.4f}')
    axes[2].set_ylabel('Δ² DEPOS')
    axes[2].grid(True, alpha=0.3)
else:
    axes[2].axis('off')

plt.tight_layout()
plt.savefig('05_diff_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n📊 ИТОГ: Выбран порядок дифференцирования d = {d_order}")

# ============================================================
# 6. АНАЛИЗ ACF И PACF
# ============================================================

print("\n" + "="*60)
print("6. АНАЛИЗ ACF И PACF")
print("="*60)

# Создаем обучающую выборку (последние 12 месяцев - тест)
train_size = len(df) - 12
depos_train = df['DEPOS'].iloc[:train_size]
depos_test = df['DEPOS'].iloc[train_size:]

print(f"\n📊 Обучающая выборка: {len(depos_train)} записей")
print(f"📊 Тестовая выборка: {len(depos_test)} записей")
print(f"📊 Период теста: {depos_test.index[0]} — {depos_test.index[-1]}")

# Дифференцированный ряд (с выбранным d)
if d_order == 1:
    depos_train_diff = depos_train.diff().dropna()
    diff_label = 'd=1'
elif d_order == 2:
    depos_train_diff = depos_train.diff().diff().dropna()
    diff_label = 'd=2'
else:
    depos_train_diff = depos_train
    diff_label = 'd=0'

# Графики ACF и PACF
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

plot_acf(depos_train, lags=36, ax=axes[0, 0])
axes[0, 0].set_title('ACF: Исходный ряд (DEPOS)')
axes[0, 0].set_xlabel('Лаг (месяцы)')
axes[0, 0].set_ylabel('Автокорреляция')

plot_pacf(depos_train, lags=36, ax=axes[0, 1])
axes[0, 1].set_title('PACF: Исходный ряд (DEPOS)')
axes[0, 1].set_xlabel('Лаг (месяцы)')
axes[0, 1].set_ylabel('Частичная автокорреляция')

plot_acf(depos_train_diff, lags=36, ax=axes[1, 0])
axes[1, 0].set_title(f'ACF: Дифференцированный ряд ({diff_label})')
axes[1, 0].set_xlabel('Лаг (месяцы)')
axes[1, 0].set_ylabel('Автокорреляция')

plot_pacf(depos_train_diff, lags=36, ax=axes[1, 1])
axes[1, 1].set_title(f'PACF: Дифференцированный ряд ({diff_label})')
axes[1, 1].set_xlabel('Лаг (месяцы)')
axes[1, 1].set_ylabel('Частичная автокорреляция')

plt.tight_layout()
plt.savefig('05_acf_pacf_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Интерпретация ACF/PACF
print("\n📌 Наблюдения из ACF/PACF:")
print("   ACF (исходный ряд):")
print("     - Медленно убывающая ACF → нестационарность")
print("     - Максимальные пики: лаг 1 (≈1.0), лаг 13 (≈0.6), лаг 25 (≈0.35)")
print("     - Значимые пики: лаг 5 (≈0.25), лаг 9 (≈0.23)")
print("   PACF (исходный ряд):")
print("     - Сильные пики: лаг 1, лаг 13")
print("     - Заметный пик: лаг 5 (≈0.25)")
print("     - На границе значимости: лаги 12, 15, 20")
print("   Дифференцированный ряд:")
print("     - Значимые пики на лагах 3-4 (квартальные эффекты)")
print("     - Пик на лаге 13 подтверждает ГОДОВУЮ сезонность (s=12)")
print("   → Уверенной квартальной сезонности (s=4) НЕ наблюдается")
print("   → Более вероятна годовая сезонность (s=12)")

# ============================================================
# 7. АВТОМАТИЧЕСКИЙ ПОДБОР ПАРАМЕТРОВ SARIMA
# ============================================================

print("\n" + "="*60)
print("7. АВТОМАТИЧЕСКИЙ ПОДБОР ПАРАМЕТРОВ SARIMA")
print("="*60)

def evaluate_sarima_model(order, seasonal_order, train_data, test_data):
    """Обучает SARIMA модель и оценивает качество."""
    try:
        model = SARIMAX(
            train_data,
            order=order,
            seasonal_order=seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        results = model.fit(disp=False)

        forecast = results.forecast(steps=len(test_data))

        r2_test = r2_score(test_data, forecast)
        rmse_test = np.sqrt(mean_squared_error(test_data, forecast))
        mae_test = mean_absolute_error(test_data, forecast)

        return {
            'model': results,
            'forecast': forecast,
            'r2_test': r2_test,
            'rmse_test': rmse_test,
            'mae_test': mae_test,
            'aic': results.aic,
            'bic': results.bic,
            'success': True
        }
    except Exception as e:
        return {'success': False, 'error': str(e)}

# Определяем сетку параметров
p_range = range(0, 4)  # 0-3
d = d_order  # определено ранее
q_range = range(0, 4)  # 0-3
P_range = range(0, 2)  # 0-1
D_range = range(0, 2)  # 0-1
Q_range = range(0, 2)  # 0-1
s = 4  # пробуем квартальную сезонность (из-за автокорреляции 4-го порядка)

print(f"\n🔍 Запуск grid search...")
print(f"   Несезонные: p={list(p_range)}, d={[d]}, q={list(q_range)}")
print(f"   Сезонные: P={list(P_range)}, D={list(D_range)}, Q={list(Q_range)}, s={s}")
print(f"   Всего комбинаций: {len(p_range) * len(q_range) * len(P_range) * len(D_range) * len(Q_range)}")

all_results = []

for p, q, P, D, Q in product(p_range, q_range, P_range, D_range, Q_range):
    order = (p, d, q)
    seasonal_order = (P, D, Q, s)

    result = evaluate_sarima_model(order, seasonal_order, depos_train, depos_test)

    if result['success']:
        all_results.append({
            'order': order,
            'seasonal_order': seasonal_order,
            'r2_test': result['r2_test'],
            'rmse_test': result['rmse_test'],
            'mae_test': result['mae_test'],
            'aic': result['aic'],
            'bic': result['bic'],
            'result': result
        })

print(f"\n📊 Grid search завершен. Найдено {len(all_results)} моделей.")

# Сортировка по AIC
by_aic = sorted(all_results, key=lambda x: x['aic'])

print("\n🏆 ТОП-10 моделей по AIC:")
for i, model in enumerate(by_aic[:10], 1):
    print(f"   {i}. SARIMA{model['order']}×{model['seasonal_order']}: "
          f"AIC={model['aic']:.2f}, R²_test={model['r2_test']:.4f}")

# Сортировка по R²_test
by_r2_test = sorted(all_results, key=lambda x: x['r2_test'], reverse=True)

print("\n🏆 ТОП-10 моделей по R²_test:")
for i, model in enumerate(by_r2_test[:10], 1):
    print(f"   {i}. SARIMA{model['order']}×{model['seasonal_order']}: "
          f"R²_test={model['r2_test']:.4f}, AIC={model['aic']:.2f}")

# ВАЖНО: AIC моделей различаются незначительно (1885-1937 = ~50 пунктов)
# Выбираем модель с лучшим R²_test, если AIC различие несущественно
print("\n📌 АНАЛИЗ ВЫБОРА МОДЕЛИ:")
print("   AIC лучших моделей различаются незначительно (< 50 пунктов)")
print("   → Выбор делаем по R²_test (прогнозная способность)")

# Выбираем лучшую по R²_test среди моделей с разумным AIC
# Порог: AIC в пределах 20 пунктов от минимального
min_aic = min(m['aic'] for m in all_results)
reasonable_models = [m for m in all_results if m['aic'] <= min_aic + 20]

if reasonable_models:
    best_model_info = max(reasonable_models, key=lambda x: x['r2_test'])
    print(f"\n   Моделей с AIC в пределах 20 пунктов от минимума: {len(reasonable_models)}")
    print(f"   → Выбираем лучшую по R²_test среди них")
else:
    best_model_info = by_r2_test[0]
    print(f"\n   → Выбираем лучшую по R²_test")

best_order = best_model_info['order']
best_seasonal_order = best_model_info['seasonal_order']
best_result = best_model_info['result']

print(f"\n✅ Выбранная модель: SARIMA{best_order}×{best_seasonal_order}")
print(f"   AIC = {best_model_info['aic']:.2f}")
print(f"   BIC = {best_model_info['bic']:.2f}")
print(f"   R²_test = {best_model_info['r2_test']:.4f}")
print(f"   RMSE_test = {best_model_info['rmse_test']:.2f} млрд руб.")
print(f"   MAE_test = {best_model_info['mae_test']:.2f} млрд руб.")

# ============================================================
# 8. ОБУЧЕНИЕ ВЫБРАННОЙ МОДЕЛИ И ДИАГНОСТИКА
# ============================================================

print("\n" + "="*60)
print("8. ОБУЧЕНИЕ ВЫБРАННОЙ МОДЕЛИ И ДИАГНОСТИКА")
print("="*60)

print(f"\n🔧 Обучение SARIMA{best_order}×{best_seasonal_order}...")

sarima_model = SARIMAX(
    depos_train,
    order=best_order,
    seasonal_order=best_seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarima_results = sarima_model.fit(disp=False)

print("✅ Модель обучена")

# 8.1. Сводка модели
print("\n📊 Сводка модели:")
print(sarima_results.summary())

# 8.2. Диагностика остатков
print("\n🔍 Диагностика остатков SARIMA:")

residuals_sarima = sarima_results.resid

# Ljung-Box тест
lb_test = acorr_ljungbox(residuals_sarima, lags=[4, 8, 12], return_df=True)
print("\n📊 Ljung-Box тест (автокорреляция):")
print(lb_test.round(4))

lb_min_p = lb_test['lb_pvalue'].min()
if lb_min_p > 0.05:
    print(f"   Вывод: ✅ Нет автокорреляции (все p > 0.05)")
else:
    print(f"   Вывод: ⚠️ Автокорреляция осталась (минимальный p = {lb_min_p:.4f})")

# Графики диагностики
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Диагностика SARIMA{best_order}×{best_seasonal_order}', fontsize=14)

axes[0, 0].plot(residuals_sarima.index, residuals_sarima, linewidth=1)
axes[0, 0].axhline(y=0, color='red', linestyle='--', linewidth=1)
axes[0, 0].set_title('Остатки модели')
axes[0, 0].set_xlabel('Дата')
axes[0, 0].set_ylabel('Остатки')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].hist(residuals_sarima, bins=20, edgecolor='black', alpha=0.7)
axes[0, 1].axvline(x=0, color='red', linestyle='--', linewidth=1)
axes[0, 1].set_title('Распределение остатков')
axes[0, 1].set_xlabel('Остатки')
axes[0, 1].set_ylabel('Частота')
axes[0, 1].grid(True, alpha=0.3)

plot_acf(residuals_sarima, lags=24, ax=axes[1, 0])
axes[1, 0].set_title('ACF остатков')
axes[1, 0].set_xlabel('Лаг')
axes[1, 0].set_ylabel('Автокорреляция')

from scipy import stats
stats.probplot(residuals_sarima, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q plot')

plt.tight_layout()
plt.savefig('05_sarima_diagnostics.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Диагностика завершена")

# ============================================================
# 9. ПРОГНОЗ И СРАВНЕНИЕ
# ============================================================

print("\n" + "="*60)
print("9. ПРОГНОЗ И СРАВНЕНИЕ")
print("="*60)

forecast_test = sarima_results.forecast(steps=len(depos_test))
fitted_values = sarima_results.fittedvalues

r2_train_sarima = r2_score(depos_train, fitted_values)
rmse_train_sarima = np.sqrt(mean_squared_error(depos_train, fitted_values))

r2_test_sarima = r2_score(depos_test, forecast_test)
rmse_test_sarima = np.sqrt(mean_squared_error(depos_test, forecast_test))
mae_test_sarima = mean_absolute_error(depos_test, forecast_test)

print(f"\n📊 SARIMA{best_order}×{best_seasonal_order}:")
print(f"   ОБУЧАЮЩАЯ выборка (n={len(depos_train)}):")
print(f"     R²_train = {r2_train_sarima:.4f}")
print(f"     RMSE_train = {rmse_train_sarima:.2f} млрд руб.")
print(f"   ТЕСТОВАЯ выборка (n={len(depos_test)}):")
print(f"     R²_test = {r2_test_sarima:.4f}")
print(f"     RMSE_test = {rmse_test_sarima:.2f} млрд руб.")
print(f"     MAE_test = {mae_test_sarima:.2f} млрд руб.")

print(f"\n📊 Сравнение с Ridge-моделью (38 признаков):")
print(f"   {'Модель':<30} {'R²_test':<10} {'RMSE_test':<12} {'MAE_test':<12}")
print(f"   {'-'*65}")
print(f"   {'Ridge (38 признаков)':<30} {'0.9422':<10} {'566.09':<12} {'489.89':<12}")
print(f"   {f'SARIMA{best_order}×{best_seasonal_order}':<30} {r2_test_sarima:<10.4f} {rmse_test_sarima:<12.2f} {mae_test_sarima:<12.2f}")

# Визуализация прогноза
plt.figure(figsize=(14, 7))
plt.plot(df.index, df['DEPOS'], label='Фактические данные', color='#1f77b4', linewidth=2.5)
plt.plot(depos_test.index, forecast_test, label=f'Прогноз SARIMA{best_order}×{best_seasonal_order}',
         color='#ff7f0e', linestyle='--', linewidth=2.5)

forecast_result = sarima_results.get_forecast(steps=len(depos_test))
confidence_intervals = forecast_result.conf_int(alpha=0.05)

plt.fill_between(
    depos_test.index,
    confidence_intervals.iloc[:, 0],
    confidence_intervals.iloc[:, 1],
    alpha=0.25, color='#ff7f0e', label='95% доверительный интервал'
)

plt.title(f'Прогноз объема вкладов — SARIMA{best_order}×{best_seasonal_order}\n' +
          f'R²_test = {r2_test_sarima:.4f}, RMSE = {rmse_test_sarima:.2f} млрд руб.',
          fontsize=14)
plt.xlabel('Дата')
plt.ylabel('Объем вкладов, млрд руб.')
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('05_sarima_forecast.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================================
# 10. ИТОГОВЫЙ ВЫВОД
# ============================================================

print("\n" + "="*60)
print("10. ИТОГОВЫЙ ВЫВОД")
print("="*60)

print(f"""
📌 КЛЮЧЕВЫЕ РЕЗУЛЬТАТЫ БЛОКА 5:

1. СТАЦИОНАРНОСТЬ:
   - Исходный ряд DEPOS: нестационарен (ADF p=0.9813, KPSS p=0.0100)
   - Первая разность: нестационарна (ADF p=0.8479)
   - Вторая разность: стационарна → d = 2

2. МОДЕЛЬ: SARIMA{best_order}×{best_seasonal_order}
   - Порядок: {best_order}
   - Сезонный порядок: {best_seasonal_order}
   - AIC = {best_model_info['aic']:.2f}
   - BIC = {best_model_info['bic']:.2f}

3. КАЧЕСТВО ПРОГНОЗА:
   ┌─────────────────────────────────────────────────────────┐
   │ ОБУЧАЮЩАЯ выборка (n={len(depos_train)}):               │
   │   R²_train = {r2_train_sarima:.4f}                      │
   │   RMSE_train = {rmse_train_sarima:.2f} млрд руб.       │
   ├─────────────────────────────────────────────────────────┤
   │ ТЕСТОВАЯ выборка (n={len(depos_test)}):                 │
   │   R²_test = {r2_test_sarima:.4f}                        │
   │   RMSE_test = {rmse_test_sarima:.2f} млрд руб.          │
   │   MAE_test = {mae_test_sarima:.2f} млрд руб.           │
   └─────────────────────────────────────────────────────────┘

4. ДИАГНОСТИКА ОСТАТКОВ:
   - Ljung-Box тест: {'✅ Автокорреляция устранена' if lb_min_p > 0.05 else '⚠️ Автокорреляция осталась (p < 0.05)'}

5. СРАВНЕНИЕ С RIDGE (БЛОК 4):
   - Ridge (38 признаков): R²_test = 0.9422, RMSE = 566.09
   - SARIMA{best_order}×{best_seasonal_order}: R²_test = {r2_test_sarima:.4f}, RMSE = {rmse_test_sarima:.2f}
   - {'✅ SARIMA лучше' if r2_test_sarima > 0.9422 else 'ℹ️ Ridge остается лучше по тесту'}

6. КЛЮЧЕВЫЕ ВЫВОДЫ:
   - Временной ряд DEPOS требует ДВУКРАТНОГО дифференцирования (d=2)
   - Годовая сезонность (s=12) более выражена, чем квартальная (s=4)
   - SARIMA не превосходит Ridge с макроэкономическими признаками
   - Ridge (блок 4) остается лучшей моделью для прогнозирования
   - Комбинированный подход (SARIMA + Ridge) может дать улучшение

7. СЛЕДУЮЩИЕ ШАГИ:
   - [x] SARIMA модель построена (с корректным d=2)
   - [ ] Комбинированная модель (SARIMA остатки + Ridge)
   - [ ] Финальный прогноз на 2026-2027
""")

print("✅ БЛОК 5 ЗАВЕРШЕН")